# Joystick IMU: perceptrón multicapa para la dirección (modo avanzado)
Mismo preprocesamiento que el softmax del modo básico (reetiquetado, 13 features sin magnetómetro,
ventanas de 20 muestras y posturas sostenidas). Cambia solo el modelo: una red con una capa oculta,
programada desde cero con NumPy.

In [ ]:
# Cargar CSV desde Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np, pandas as pd

RUTA = '/content/drive/MyDrive/Sistemas inteligentes/Regresión Softmax Joystick/ensayos_imu.csv'
df = pd.read_csv(RUTA, encoding='latin1')

# Unificar 'atrás' (tilde dañada) -> 'atras'
df['Direccion'] = df['Direccion'].astype(str).str.strip()
df.loc[df['Direccion'].str.lower().str.startswith('atr'), 'Direccion'] = 'atras'

# Un id único por ensayo
df['grupo'] = df['user_id'].astype(int) * 100000 + df['trial_id'].astype(int)
df = df.sort_values(['grupo', 'sample_id']).reset_index(drop=True)

print('Muestras:', len(df), '| Ensayos:', df['grupo'].nunique())
print(df['Direccion'].value_counts())

In [ ]:
# Reetiquetado: el inicio quieto de cada ensayo pasa a 'N'
# Se reconstruye el ángulo con el mismo filtro complementario de receptorDB2.py
ANG_MAX = {'adelante': 50.0, 'atras': 35.0, 'izquierda': 33.0, 'derecha': 30.0}
ALPHA, DT, N0 = 0.98, 0.01, 10   # filtro, 100 Hz, muestras iniciales de referencia
FRAC, MIN_DEG = 0.30, 3.0        # umbral = max(3°, 30% del ángulo objetivo)

def delta_angulos(g):
    ax, ay, az = g['a_x'].values, g['a_y'].values, g['a_z'].values
    gy, gz = g['w_y'].values, g['w_z'].values
    acc_y = np.degrees(np.arctan2(ax, np.sqrt(ay**2 + az**2)))
    acc_z = np.degrees(np.arctan2(az, np.sqrt(ax**2 + ay**2)))
    off_y, off_z = gy[:N0].mean(), gz[:N0].mean()   # sesgo del giro
    y, z = acc_y[0], acc_z[0]
    ang_y, ang_z = np.empty(len(g)), np.empty(len(g))
    for i in range(len(g)):
        y = ALPHA * (y + (gy[i] - off_y) * DT) + (1 - ALPHA) * acc_y[i]
        z = ALPHA * (z + (gz[i] - off_z) * DT) + (1 - ALPHA) * acc_z[i]
        ang_y[i], ang_z[i] = y, z
    return np.abs(ang_y - ang_y[0]), np.abs(ang_z - ang_z[0])

df['Direccion_orig'] = df['Direccion']
nueva = df['Direccion'].copy()
alcance = []
for gid, g in df.groupby('grupo'):
    d = g['Direccion'].iloc[0]
    if d not in ANG_MAX:
        continue
    dy, dz = delta_angulos(g)
    delta = dz if d in ('izquierda', 'derecha') else dy
    objetivo = ANG_MAX[d] * g['velocidad_objetivo'].iloc[0]
    nueva.loc[g.index[delta < max(MIN_DEG, FRAC * objetivo)]] = 'N'
    alcance.append(delta.max() / objetivo)
df['Direccion'] = nueva

# Control: ~1.0 significa que el ángulo reconstruido llega al objetivo
print('Ángulo máx / objetivo (mediana):', round(float(np.median(alcance)), 2))
print('\nAntes:\n', df['Direccion_orig'].value_counts())
print('\nDespués:\n', df['Direccion'].value_counts())

In [ ]:
# Features por muestra (13), sin magnetómetro: el modelo no depende de hacia dónde mire el usuario
N_BASE = 13
NOMBRES_BASE = ['a_x','a_y','a_z','w_x','w_y','w_z',
                'a_mag','w_mag','inc_y','inc_z','ax_ay','ax_az','ay_az']

def features_base(a, w, m, mref):   # m y mref se reciben pero no se usan
    ax, ay, az = a.T; wx, wy, wz = w.T
    a_mag = np.sqrt(ax**2 + ay**2 + az**2)
    w_mag = np.sqrt(wx**2 + wy**2 + wz**2)
    inc_y = np.arctan2(ay, np.sqrt(ax**2 + az**2))   # inclinación lateral
    inc_z = np.arctan2(az, np.sqrt(ax**2 + ay**2))   # inclinación adelante/atrás
    return np.column_stack([ax, ay, az, wx, wy, wz,
                            a_mag, w_mag, inc_y, inc_z, ax*ay, ax*az, ay*az])

In [ ]:
# Ventanas de 20 muestras (200 ms), paso 5, sin mezclar ensayos
# Por ventana: media, desviación (ddof=0, igual que el ESP32), mínimo y máximo -> 68 features
WINDOW_SIZE, STEP_SIZE = 20, 5

Xw, yw, gw = [], [], []
for gid, g in df.groupby('grupo'):
    a = g[['a_x','a_y','a_z']].values
    w = g[['w_x','w_y','w_z']].values
    m = g[['m_x','m_y','m_z']].values
    F = features_base(a, w, m, m[:N0].mean(0))
    lab = g['Direccion'].values
    for s in range(0, len(g) - WINDOW_SIZE + 1, STEP_SIZE):
        v = F[s:s + WINDOW_SIZE]
        Xw.append(np.concatenate([v.mean(0), v.std(0), v.min(0), v.max(0)]))
        yw.append(pd.Series(lab[s:s + WINDOW_SIZE]).mode()[0])   # etiqueta por mayoría
        gw.append(gid)

Xw, yw, gw = np.array(Xw), np.array(yw), np.array(gw)
NOMBRES = [f'{e}_{n}' for e in ['mean','std','min','max'] for n in NOMBRES_BASE]
print('Ventanas:', Xw.shape)
print(pd.Series(yw).value_counts())

In [ ]:
# Agrega ventanas de postura sostenida: los ensayos terminan en movimiento y nunca muestran la mano quieta inclinada
N_SOST, PICO_MIN, G = 3, 10.0, 9.80665   # ventanas por ensayo, ángulo mínimo del pico (°), gravedad
rng = np.random.default_rng(0)

Xs, ys, gs = [], [], []
for gid, g in df.groupby('grupo'):
    d = g['Direccion_orig'].iloc[0]
    if d not in ANG_MAX:
        continue
    dy, dz = delta_angulos(g)
    delta = dz if d in ('izquierda', 'derecha') else dy
    if delta.max() < PICO_MIN:          # inclinaciones muy pequeñas se confunden con N
        continue
    a = g[['a_x','a_y','a_z']].values
    w = g[['w_x','w_y','w_z']].values
    m = g[['m_x','m_y','m_z']].values
    mref = m[:N0].mean(0)
    quieto = g['Direccion'].values == 'N'   # inicio quieto: ruido real del sensor
    if quieto.sum() < WINDOW_SIZE:
        continue
    pico = delta >= 0.9 * delta.max()
    a_p = a[pico].mean(0)
    a_p = a_p / np.linalg.norm(a_p) * G   # quieto: solo gravedad, sin aceleración del movimiento
    m_p = m[pico].mean(0)                 # orientación del magnetómetro en el pico
    ra, rw, rm = a[quieto], w[quieto], m[quieto]
    for k in range(N_SOST):
        i = rng.choice(len(ra), WINDOW_SIZE)
        aa = a_p + (ra[i] - ra.mean(0))     # postura del pico + ruido en reposo
        ww = rw[i]                          # giro ~0 (mano quieta)
        mm = m_p + (rm[i] - rm.mean(0))
        v = features_base(aa, ww, mm, mref)
        Xs.append(np.concatenate([v.mean(0), v.std(0), v.min(0), v.max(0)]))
        ys.append(d)
        gs.append(gid)                      # mismo grupo: no se filtra entre train y test

tipo = np.array(['orig'] * len(Xw) + ['sost'] * len(Xs))
Xw = np.vstack([Xw, np.array(Xs)])
yw = np.concatenate([yw, ys])
gw = np.concatenate([gw, gs])
print('Ventanas sostenidas agregadas:', len(Xs), '| Total:', Xw.shape)
print(pd.Series(ys).value_counts())

In [ ]:
# División por ensayo (sin fuga): train 64% / validación 16% / test 20%
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder().fit(yw)
y = le.transform(yw)
CLASES = list(le.classes_)

i_trval, i_te = next(GroupShuffleSplit(1, test_size=0.20, random_state=42).split(Xw, y, gw))
i_tr, i_va = next(GroupShuffleSplit(1, test_size=0.20, random_state=42).split(Xw[i_trval], y[i_trval], gw[i_trval]))
i_tr, i_va = i_trval[i_tr], i_trval[i_va]

scaler = StandardScaler().fit(Xw[i_tr])   # normalización z-score (solo con train)
Xtr, Xva, Xte = (scaler.transform(Xw[i]) for i in (i_tr, i_va, i_te))
ytr, yva, yte = y[i_tr], y[i_va], y[i_te]

print('Clases:', dict(enumerate(CLASES)))
print(f'Train: {len(ytr)} | Validación: {len(yva)} | Test: {len(yte)} ventanas')

In [ ]:
# Red 52 -> 8 -> 5
# Capa oculta: tanh | Salida: softmax (probabilidad de cada dirección)
OCULTAS = 8
K, F = len(CLASES), Xtr.shape[1]
rng = np.random.default_rng(0)
W1 = rng.normal(0, np.sqrt(1 / F), (F, OCULTAS)); b1 = np.zeros(OCULTAS)
W2 = rng.normal(0, np.sqrt(1 / OCULTAS), (OCULTAS, K)); b2 = np.zeros(K)

def softmax(Z):
    Z = Z - Z.max(1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(1, keepdims=True)

def adelante(X):
    h = np.tanh(X @ W1 + b1)        # capa oculta
    return h, softmax(h @ W2 + b2)  # salida

predecir = lambda X: adelante(X)[1].argmax(1)

In [ ]:
# Entrenamiento con gradiente descendente y retropropagación (costo: entropía cruzada con balance de clases y regularización L2)
EPOCAS, LR, LAMBDA = 3000, 0.3, 1e-2
Y = np.eye(K)[ytr]
peso = (len(ytr) / (K * np.bincount(ytr, minlength=K)))[ytr]
train_accs, val_accs, losses = [], [], []

for ep in range(EPOCAS):
    h, P = adelante(Xtr)
    d2 = (P - Y) * peso[:, None] / peso.sum()     # error en la salida
    d1 = (d2 @ W2.T) * (1 - h**2)                 # error que pasa a la capa oculta (derivada de tanh)
    W2 -= LR * (h.T @ d2 + LAMBDA * W2);  b2 -= LR * d2.sum(0)
    W1 -= LR * (Xtr.T @ d1 + LAMBDA * W1); b1 -= LR * d1.sum(0)
    losses.append(-(peso * np.log(P[np.arange(len(ytr)), ytr] + 1e-12)).sum() / peso.sum())
    train_accs.append((P.argmax(1) == ytr).mean())
    val_accs.append((predecir(Xva) == yva).mean())

print(f'Train acc final: {train_accs[-1]:.4f} | Val acc final: {val_accs[-1]:.4f}')

In [ ]:
# Accuracy por iteración y matrices de confusión
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

plt.figure(figsize=(9, 4))
plt.plot(train_accs, label='Train')
plt.plot(val_accs, label='Validación')
plt.xlabel('Iteración'); plt.ylabel('Accuracy'); plt.grid(alpha=.4); plt.legend()
plt.title('Evolución del accuracy - MLP dirección'); plt.show()

for nombre, X_, y_ in [('Train', Xtr, ytr), ('Test', Xte, yte)]:
    p = predecir(X_)
    print(f'\n===== {nombre}: accuracy = {accuracy_score(y_, p):.4f}')
    print(classification_report(y_, p, target_names=CLASES, digits=3))
    plt.figure(figsize=(6, 5))
    sns.heatmap(confusion_matrix(y_, p), annot=True, fmt='d', cmap='Greys',
                xticklabels=CLASES, yticklabels=CLASES)
    plt.xlabel('Predicción'); plt.ylabel('Real'); plt.title(f'Matriz de confusión - {nombre}')
    plt.show()

In [ ]:
# Desempeño en test por tipo de ventana (originales vs. posturas sostenidas)
from sklearn.metrics import f1_score
p, t_te = predecir(Xte), tipo[i_te]
for nombre, s in [('Total', t_te != ''), ('Originales', t_te == 'orig'), ('Sostenidas', t_te == 'sost')]:
    print(f'{nombre:11s} n={s.sum():4d}  accuracy={accuracy_score(yte[s], p[s]):.3f}  F1 macro={f1_score(yte[s], p[s], average="macro"):.3f}')
s = t_te == 'sost'
print('\nPosturas sostenidas (filas = real, columnas = predicción):')
print(pd.crosstab(np.array(CLASES)[yte[s]], np.array(CLASES)[p[s]], rownames=['real'], colnames=['pred']))

In [ ]:
# Pesos y sesgos (parámetros θ) y ecuación del modelo
np.set_printoptions(precision=5, suppress=True, linewidth=150)
print('W1', W1.shape, '\n', W1, '\nb1', b1)
print('W2', W2.shape, '\n', W2, '\nb2', b2)
print('\nModelo:')
print(f'  x~ = (x - mu) / sigma                 {F} features por ventana')
print(f'  h  = tanh(x~ W1 + b1)                 {OCULTAS} neuronas')
print('  P(y=k | x) = softmax(h W2 + b2)_k')
print('  dirección = argmax_k P(y=k | x)')

In [ ]:
# Bloque para pegar en el .ino
f = lambda v: ', '.join(f'{x:.7f}f' for x in np.ravel(v))
mat = lambda nombre, M: f'const float {nombre}[{M.shape[0]}][{M.shape[1]}] = {{\n' + \
      ',\n'.join('  { ' + f(fila) + ' }' for fila in M) + '\n};'
bloque = '\n'.join([
    '// Generado por el notebook MLP dirección. No editar a mano.',
    f'const int MD_OCULTAS = {OCULTAS};',
    f'const float MD_MEAN[{F}] = {{ {f(scaler.mean_)} }};',
    f'const float MD_SCALE[{F}] = {{ {f(scaler.scale_)} }};',
    mat('MD_W1', W1), f'const float MD_B1[{OCULTAS}] = {{ {f(b1)} }};',
    mat('MD_W2', W2), f'const float MD_B2[{K}] = {{ {f(b2)} }};',
])
print(bloque)
with open('parametros_mlp_direccion.txt', 'w') as fh:
    fh.write(bloque + '\n')
from google.colab import files
files.download('parametros_mlp_direccion.txt')